# Chapter 8: Knowledge Distillation — Making Powerful Models Practical

# Required Libraries

In [1]:
# Import PyTorch, the core deep learning framework used for tensor operations
# and automatic differentiation (autograd)
import torch

# Import PyTorch's neural network module, which provides layers such as
# Linear, Conv2d, LayerNorm, and other building blocks for neural networks
import torch.nn as nn

# Import the functional API, which contains stateless functions for operations
# such as activation functions, loss functions, and pooling
import torch.nn.functional as F

# Import torchvision, a library that provides popular computer vision datasets,
# pre-trained models, and image processing utilities
import torchvision

# Import image transformation utilities used for preprocessing images,
# such as resizing, normalization, and data augmentation
import torchvision.transforms as transforms

# Import DataLoader, which efficiently loads data in batches, supports
# shuffling, and enables parallel data loading during training
from torch.utils.data import DataLoader

# Temperature and Dark Knowledge

Under a standard softmax (T=1), the teacher's output is too peaked — dark knowledge is hidden in near-zero probabilities. **Temperature scaling** solves this by dividing logits by T before softmax:

$$q_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- **T=1**: Standard softmax (too peaked)
- **T=2–5**: The Goldilocks zone (dark knowledge emerges)
- **T≥10**: Too soft (signal lost in near-uniformity)

### Listing 8.1: Visualizing Temperature-Scaled Softmax

In [2]:
# Define a helper function to visualize the effect of different
# temperature values on the softmax probability distribution
def visualize_temperature(logits, temps):

    # Documentation string explaining the purpose of the function
    # Temperature controls how confident or random the output probabilities are
    """Show how temperature affects softmax."""

    # Iterate through each temperature value provided in the list
    for T in temps:

        # Scale the logits by the temperature and apply softmax
        #
        # T = 1   → Original probability distribution
        # T > 1   → Softer, more uniform probabilities
        # T < 1   → Sharper, more confident probabilities
        probs = F.softmax(logits / T, dim=-1)

        # Display the temperature and corresponding probabilities
        # rounded to 3 decimal places for easier reading
        print(f"T={T:2d}: {[f'{p:.3f}' for p in probs.tolist()]}")

# Create a sample logits vector representing raw model scores
#
# Token 1 score = 5.0 (highest preference)
# Token 2 score = 3.0
# Token 3 score = 1.0
# Token 4 score = -1.0 (lowest preference)
logits = torch.tensor([5.0, 3.0, 1.0, -1.0])

# Visualize how the probability distribution changes as
# temperature increases from 1 to 10
visualize_temperature(logits, [1, 2, 3, 5, 10])

T= 1: ['0.865', '0.117', '0.016', '0.002']
T= 2: ['0.644', '0.237', '0.087', '0.032']
T= 3: ['0.523', '0.268', '0.138', '0.071']
T= 5: ['0.413', '0.277', '0.186', '0.124']
T=10: ['0.329', '0.270', '0.221', '0.181']


# Listing 8.2: The Knowledge Distillation Loss Function

In [3]:
# Define the knowledge distillation loss function
#
# This function combines:
# 1. Hard targets  → Ground-truth labels from the dataset
# 2. Soft targets  → Probability distribution produced by the teacher model
#
# The student learns both the correct answer and the teacher's reasoning pattern.
def distillation_loss(student_logits, teacher_logits, labels,
                      temperature=3.0, alpha=0.3):

    # Documentation string describing the purpose of the function
    """Complete knowledge distillation loss combining hard and soft targets."""

    # Convert teacher logits into a softened probability distribution.
    #
    # Dividing by temperature makes the probability distribution smoother,
    # revealing relationships between classes that are hidden in normal softmax.
    soft_teacher = F.softmax(teacher_logits / temperature, dim=-1)

    # Convert student logits into log probabilities using the same temperature.
    #
    # KL divergence expects log probabilities for the student distribution.
    soft_student = F.log_softmax(student_logits / temperature, dim=-1)

    # Compute KL Divergence between the student and teacher distributions.
    #
    # This measures how different the student's predictions are from
    # the teacher's predictions.
    #
    # Lower KL divergence means the student is imitating the teacher well.
    kl_loss = F.kl_div(
        soft_student,
        soft_teacher,
        reduction='batchmean'
    )

    # Compute standard cross-entropy loss using the true labels.
    #
    # This ensures the student still learns the correct answers from
    # the training dataset.
    hard_loss = F.cross_entropy(student_logits, labels)

    # Combine hard-label loss and soft-label loss.
    #
    # alpha controls the balance:
    # alpha = 1.0 → only hard labels
    # alpha = 0.0 → only teacher guidance
    #
    # temperature² is included as recommended in Hinton's
    # Knowledge Distillation paper to properly scale gradients.
    return (
        alpha * hard_loss
        + (1 - alpha) * temperature**2 * kl_loss
    )

In [4]:
# Set a fixed random seed so that the generated random values
# remain the same every time the code is executed.
#
# This helps make experiments reproducible.
torch.manual_seed(42)

# Generate random logits produced by the student model.
#
# Shape: (4, 10)
# 4  -> batch size (4 training examples)
# 10 -> number of classes
student_logits = torch.randn(4, 10)

# Generate random logits produced by the teacher model.
#
# In a real distillation setup, these would come from a large,
# pre-trained teacher network.
teacher_logits = torch.randn(4, 10)

# Generate random ground-truth class labels.
#
# Each label is an integer between 0 and 9.
# Shape: (4,)
labels = torch.randint(0, 10, (4,))

# Compute the complete distillation loss.
#
# temperature=3.0:
#     Produces softer probability distributions.
#
# alpha=0.3:
#     30% weight on hard-label learning
#     70% weight on teacher knowledge transfer
loss = distillation_loss(
    student_logits,
    teacher_logits,
    labels,
    temperature=3.0,
    alpha=0.3
)

# Display the final combined loss value.
print(f"Distillation loss: {loss.item():.4f}")

# Compute the standard cross-entropy loss using only
# the ground-truth labels.
#
# This represents traditional supervised learning
# without knowledge distillation.
hard_only = F.cross_entropy(student_logits, labels)

# Display the hard-label loss.
print(f"Hard loss only:    {hard_only.item():.4f}")

# Explain the purpose of the comparison.
#
# Hard loss:
#     Learns directly from dataset labels.
#
# Distillation loss:
#     Learns from both labels and teacher predictions.
print(f"\nThe distillation loss combines hard labels (grounding) with")
print(f"soft targets (dark knowledge transfer).")

Distillation loss: 1.5987
Hard loss only:    3.4429

The distillation loss combines hard labels (grounding) with
soft targets (dark knowledge transfer).


# DeepSeek-R1's Distillation Recipe

# Implementing Classical Knowledge Distillation in PyTorch

# The Teacher and Student Architectures

- **Teacher**: A larger CNN (3 conv layers, 512-unit FC layer) — ~435K parameters
- **Student**: A smaller CNN (2 conv layers, 128-unit FC layer) — ~46K parameters
- **Compression ratio**: ~9.5x

# Listing 8.3: The DistillationTrainer Class

In [5]:
# Define the Teacher CNN.
#
# The teacher is a larger and more powerful model that learns
# rich feature representations from the training data.
#
# During knowledge distillation, the teacher's predictions are
# used to guide the training of a smaller student model.
class TeacherCNN(nn.Module):

    # Documentation string describing the model
    """Larger CNN for CIFAR-10 classification (the teacher)."""

    # Initialize the network architecture
    def __init__(self, num_classes=10):

        # Initialize the parent nn.Module class
        super().__init__()

        # Feature extraction layers
        #
        # These convolutional layers learn spatial patterns such as:
        # - edges
        # - textures
        # - shapes
        # - object parts
        self.features = nn.Sequential(

            # First convolution layer
            # Input: 3-channel RGB image
            # Output: 64 feature maps
            nn.Conv2d(3, 64, 3, padding=1),

            # Normalize activations for faster and more stable training
            nn.BatchNorm2d(64),

            # Apply ReLU non-linearity
            nn.ReLU(),

            # Reduce spatial dimensions by a factor of 2
            nn.MaxPool2d(2),

            # Second convolution layer
            # Increase feature maps from 64 → 128
            nn.Conv2d(64, 128, 3, padding=1),

            # Batch normalization
            nn.BatchNorm2d(128),

            # ReLU activation
            nn.ReLU(),

            # Downsample feature maps
            nn.MaxPool2d(2),

            # Third convolution layer
            # Increase representational capacity
            nn.Conv2d(128, 256, 3, padding=1),

            # Normalize activations
            nn.BatchNorm2d(256),

            # Non-linear activation
            nn.ReLU(),

            # Convert feature maps to fixed size (4 × 4)
            # regardless of input dimensions
            nn.AdaptiveAvgPool2d(4),
        )

        # Classification head
        #
        # Converts extracted image features into class predictions.
        self.classifier = nn.Sequential(

            # Flattened size:
            # 256 channels × 4 × 4 spatial locations
            nn.Linear(256 * 4 * 4, 512),

            # Non-linearity
            nn.ReLU(),

            # Dropout for regularization
            nn.Dropout(0.3),

            # Final classification layer
            nn.Linear(512, num_classes),
        )

    # Define forward propagation
    def forward(self, x):

        # Extract image features
        x = self.features(x)

        # Flatten feature maps into a vector
        #
        # Shape:
        # (batch, channels, height, width)
        #        ↓
        # (batch, features)
        x = x.view(x.size(0), -1)

        # Generate class logits
        return self.classifier(x)


# ============================================================
# Student CNN
# ============================================================
#
# The student model is intentionally smaller and faster.
#
# Goal:
# Achieve performance close to the teacher while using
# fewer parameters and less computation.
class StudentCNN(nn.Module):

    # Documentation string describing the model
    """Smaller CNN for CIFAR-10 classification (the student)."""

    # Initialize the student architecture
    def __init__(self, num_classes=10):

        # Initialize parent class
        super().__init__()

        # Lightweight feature extractor
        self.features = nn.Sequential(

            # First convolution layer
            #
            # Uses only 32 filters instead of 64,
            # reducing parameter count.
            nn.Conv2d(3, 32, 3, padding=1),

            # Batch normalization
            nn.BatchNorm2d(32),

            # ReLU activation
            nn.ReLU(),

            # Downsampling
            nn.MaxPool2d(2),

            # Second convolution layer
            #
            # Uses 64 filters instead of teacher's
            # deeper 128→256 progression.
            nn.Conv2d(32, 64, 3, padding=1),

            # Batch normalization
            nn.BatchNorm2d(64),

            # ReLU activation
            nn.ReLU(),

            # Fixed-size output feature map
            nn.AdaptiveAvgPool2d(4),
        )

        # Lightweight classifier
        self.classifier = nn.Sequential(

            # Flattened feature size:
            # 64 × 4 × 4
            nn.Linear(64 * 4 * 4, 128),

            # ReLU activation
            nn.ReLU(),

            # Dropout for regularization
            nn.Dropout(0.3),

            # Final prediction layer
            nn.Linear(128, num_classes),
        )

    # Define forward propagation
    def forward(self, x):

        # Extract image features
        x = self.features(x)

        # Flatten feature maps
        x = x.view(x.size(0), -1)

        # Produce class logits
        return self.classifier(x)

In [6]:
# Define a trainer class that manages the complete
# Knowledge Distillation (KD) training process.
#
# Responsibilities:
# 1. Store teacher and student models
# 2. Configure distillation hyperparameters
# 3. Perform training steps
# 4. Update student weights
# 5. Train across multiple batches and epochs
class DistillationTrainer:

    # Documentation string describing the purpose of the class
    """Manages the complete knowledge distillation pipeline."""

    # Initialize the trainer
    def __init__(
        self,
        teacher,
        student,
        temperature=3.0,
        alpha=0.3,
        lr=1e-3
    ):

        # Store the teacher model
        self.teacher = teacher

        # Store the student model
        self.student = student

        # Temperature controls the softness of teacher predictions
        #
        # Higher temperature → smoother probability distribution
        self.temperature = temperature

        # Alpha controls the balance between:
        # - hard label learning
        # - teacher-guided learning
        self.alpha = alpha

        # Detect the device (CPU or GPU) being used
        self.device = next(student.parameters()).device

        # Create an Adam optimizer for updating only
        # the student model parameters
        self.optimizer = torch.optim.Adam(
            student.parameters(),
            lr=lr
        )

        # ----------------------------------------------------
        # Freeze the teacher model
        # ----------------------------------------------------
        #
        # The teacher acts as a fixed source of knowledge.
        #
        # We do not want its weights to change during
        # student training.
        self.teacher.eval()

        # Disable gradient computation for all teacher parameters
        for param in self.teacher.parameters():
            param.requires_grad = False

    # --------------------------------------------------------
    # Single Training Step
    # --------------------------------------------------------
    #
    # Performs:
    # Teacher Forward Pass
    # Student Forward Pass
    # Distillation Loss Computation
    # Backpropagation
    # Weight Update
    def train_step(self, images, labels):

        # Documentation string
        """One training step: forward pass through both models, compute loss, update student."""

        # Put the student model into training mode
        #
        # Enables dropout and batch normalization updates.
        self.student.train()

        # Move data to the correct device (CPU/GPU)
        images = images.to(self.device)
        labels = labels.to(self.device)

        # ----------------------------------------------------
        # Teacher Forward Pass
        # ----------------------------------------------------
        #
        # No gradients are needed because the teacher
        # is frozen.
        with torch.no_grad():

            # Generate teacher predictions
            teacher_logits = self.teacher(images)

        # ----------------------------------------------------
        # Student Forward Pass
        # ----------------------------------------------------
        #
        # Generate student predictions.
        student_logits = self.student(images)

        # ----------------------------------------------------
        # Compute Distillation Loss
        # ----------------------------------------------------
        #
        # Combines:
        # - Cross Entropy Loss (hard labels)
        # - KL Divergence Loss (teacher knowledge)
        loss = distillation_loss(
            student_logits,
            teacher_logits,
            labels,
            temperature=self.temperature,
            alpha=self.alpha
        )

        # ----------------------------------------------------
        # Update Student Weights
        # ----------------------------------------------------

        # Clear old gradients from the previous batch
        self.optimizer.zero_grad()

        # Compute gradients using backpropagation
        loss.backward()

        # Update student parameters
        self.optimizer.step()

        # Return the scalar loss value
        return loss.item()

    # --------------------------------------------------------
    # Train One Full Epoch
    # --------------------------------------------------------
    #
    # An epoch means processing every training sample once.
    def train_epoch(self, dataloader):

        # Documentation string
        """Train for one full epoch."""

        # Store cumulative loss
        total_loss = 0.0

        # Iterate through all batches
        for images, labels in dataloader:

            # Train on one batch and accumulate loss
            total_loss += self.train_step(
                images,
                labels
            )

        # Compute average loss across all batches
        return total_loss / len(dataloader)

### Listing 8.4 & 8.5: Running Distillation on CIFAR-10

The complete pipeline:
1. Load CIFAR-10 data
2. Train the teacher model
3. Distill the teacher's knowledge into the student
4. Train a baseline student from scratch for comparison
5. Evaluate all three models

In [7]:
# ============================================================
# Model Evaluation Function
# ============================================================
#
# This function measures how well a trained model performs
# on a dataset by computing classification accuracy.
#
# Accuracy =
# (Number of Correct Predictions / Total Samples) × 100
def evaluate(model, dataloader, device):

    # Documentation string describing the function
    """Evaluate a model on a dataset and return accuracy."""

    # Put the model into evaluation mode.
    #
    # This disables training-specific behaviors such as:
    # - Dropout randomness
    # - BatchNorm statistics updates
    model.eval()

    # Initialize counters
    #
    # correct → number of correct predictions
    # total   → total number of samples processed
    correct, total = 0, 0

    # Disable gradient computation.
    #
    # During evaluation we only need predictions,
    # so gradient calculation would waste memory and time.
    with torch.no_grad():

        # Process each batch in the dataset
        for images, labels in dataloader:

            # Move batch data to the selected device
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass through the model
            outputs = model(images)

            # Find the class with the highest score
            #
            # outputs shape:
            # (batch_size, num_classes)
            #
            # max(1) returns:
            # values      → highest scores
            # predicted   → corresponding class indices
            _, predicted = outputs.max(1)

            # Add current batch size to total samples
            total += labels.size(0)

            # Count how many predictions match the true labels
            correct += predicted.eq(labels).sum().item()

    # Compute classification accuracy percentage
    return 100.0 * correct / total


# ============================================================
# Standard Supervised Training Function
# ============================================================
#
# This function trains a model from scratch using only
# hard labels from the dataset.
#
# No teacher model is involved.
# No knowledge distillation is used.
def train_from_scratch(
    model,
    dataloader,
    epochs,
    device,
    lr=1e-3
):

    # Documentation string
    """Train a model from scratch using only hard labels."""

    # Create Adam optimizer.
    #
    # Adam combines momentum and adaptive learning rates,
    # making it a popular choice for deep learning.
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    # Put model into training mode.
    #
    # Enables:
    # - Dropout
    # - BatchNorm updates
    model.train()

    # --------------------------------------------------------
    # Training Loop
    # --------------------------------------------------------
    #
    # Repeat training for the specified number of epochs.
    for epoch in range(epochs):

        # Store cumulative loss for the epoch
        total_loss = 0.0

        # Process every batch in the dataset
        for images, labels in dataloader:

            # Move batch to the selected device
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            #
            # Generate class logits.
            outputs = model(images)

            # Compute cross-entropy loss using
            # ground-truth labels.
            #
            # This is standard supervised learning.
            loss = F.cross_entropy(
                outputs,
                labels
            )

            # Clear gradients from previous batch
            optimizer.zero_grad()

            # Compute gradients via backpropagation
            loss.backward()

            # Update model parameters
            optimizer.step()

            # Add batch loss to cumulative epoch loss
            total_loss += loss.item()

        # Compute average loss across all batches
        avg_loss = total_loss / len(dataloader)

        # Display progress every 5 epochs
        if (epoch + 1) % 5 == 0:

            print(
                f"  Epoch {epoch+1}/{epochs}, "
                f"Loss: {avg_loss:.4f}"
            )

    # Return the trained model
    return model

In [8]:
# ============================================================
# Complete Knowledge Distillation Experiment
# ============================================================
#
# This function demonstrates the full teacher-student
# training pipeline on the CIFAR-10 dataset.
#
# Workflow:
#
# Step 1:
# Train a large TeacherCNN using standard supervised learning.
#
# Step 2:
# Train a smaller StudentCNN using knowledge distillation.
#
# Step 3:
# Train another StudentCNN from scratch as a baseline.
#
# Step 4:
# Compare accuracies and model sizes.
def run_distillation():

    # Select GPU if available, otherwise use CPU.
    device = torch.device(
        'cuda' if torch.cuda.is_available() else 'cpu'
    )

    # Display the selected device.
    print(f"Using device: {device}")

    # ========================================================
    # DATA PREPARATION
    # ========================================================

    # Create an image preprocessing pipeline.
    #
    # ToTensor():
    #     Converts PIL images into PyTorch tensors.
    #
    # Normalize():
    #     Standardizes pixel values using CIFAR-10 statistics.
    #
    # Normalization helps stabilize training and improves
    # optimization performance.
    transform = transforms.Compose([
        transforms.ToTensor(),

        transforms.Normalize(
            (0.4914, 0.4822, 0.4465),    # Mean values
            (0.2470, 0.2435, 0.2616)     # Standard deviations
        ),
    ])

    # --------------------------------------------------------
    # Load CIFAR-10 Training Dataset
    # --------------------------------------------------------
    trainset = torchvision.datasets.CIFAR10(
        root='./data',
        train=True,
        download=True,
        transform=transform
    )

    # --------------------------------------------------------
    # Load CIFAR-10 Test Dataset
    # --------------------------------------------------------
    testset = torchvision.datasets.CIFAR10(
        root='./data',
        train=False,
        download=True,
        transform=transform
    )

    # --------------------------------------------------------
    # Create DataLoaders
    # --------------------------------------------------------
    #
    # trainloader:
    #     Used for training.
    #
    # testloader:
    #     Used for evaluation.
    trainloader = DataLoader(
        trainset,
        batch_size=128,
        shuffle=True,
        num_workers=2
    )

    testloader = DataLoader(
        testset,
        batch_size=256,
        shuffle=False,
        num_workers=2
    )

    # ========================================================
    # STEP 1: TRAIN THE TEACHER
    # ========================================================

    print("\n" + "=" * 50)
    print("Step 1: Training the Teacher")
    print("=" * 50)

    # Create the teacher model and move it to the device.
    teacher = TeacherCNN().to(device)

    # Count total trainable parameters.
    teacher_params = sum(
        p.numel()
        for p in teacher.parameters()
    )

    print(f"Teacher parameters: {teacher_params:,}")

    # Train teacher using standard supervised learning.
    teacher = train_from_scratch(
        teacher,
        trainloader,
        epochs=20,
        device=device
    )

    # Evaluate teacher performance.
    teacher_acc = evaluate(
        teacher,
        testloader,
        device
    )

    print(f"Teacher test accuracy: {teacher_acc:.2f}%")

    # ========================================================
    # STEP 2: DISTILL KNOWLEDGE INTO STUDENT
    # ========================================================

    print("\n" + "=" * 50)
    print("Step 2: Distilling Knowledge into the Student")
    print("=" * 50)

    # Create the student model.
    distilled_student = StudentCNN().to(device)

    # Count student parameters.
    student_params = sum(
        p.numel()
        for p in distilled_student.parameters()
    )

    print(f"Student parameters: {student_params:,}")

    # Show model compression ratio.
    #
    # Example:
    # 10x means the teacher has 10 times more parameters.
    print(
        f"Compression ratio: "
        f"{teacher_params / student_params:.1f}x"
    )

    # Create the knowledge distillation trainer.
    trainer = DistillationTrainer(
        teacher=teacher,
        student=distilled_student,
        temperature=3.0,
        alpha=0.3,
        lr=1e-3
    )

    # Train the student using teacher guidance.
    for epoch in range(20):

        # Train one epoch.
        avg_loss = trainer.train_epoch(
            trainloader
        )

        # Display progress every 5 epochs.
        if (epoch + 1) % 5 == 0:

            print(
                f"  Epoch {epoch+1}/20, "
                f"Loss: {avg_loss:.4f}"
            )

    # Evaluate the distilled student.
    distilled_acc = evaluate(
        distilled_student,
        testloader,
        device
    )

    print(
        f"Distilled student test accuracy: "
        f"{distilled_acc:.2f}%"
    )

    # ========================================================
    # STEP 3: TRAIN BASELINE STUDENT
    # ========================================================

    print("\n" + "=" * 50)
    print("Step 3: Training Baseline Student (from scratch)")
    print("=" * 50)

    # Create another student model.
    #
    # This student will not receive any teacher guidance.
    baseline_student = StudentCNN().to(device)

    # Train using ordinary supervised learning.
    baseline_student = train_from_scratch(
        baseline_student,
        trainloader,
        epochs=20,
        device=device
    )

    # Evaluate baseline performance.
    baseline_acc = evaluate(
        baseline_student,
        testloader,
        device
    )

    print(
        f"Baseline student test accuracy: "
        f"{baseline_acc:.2f}%"
    )

    # ========================================================
    # STEP 4: DISPLAY RESULTS
    # ========================================================

    print("\n" + "=" * 50)
    print("RESULTS SUMMARY")
    print("=" * 50)

    # Display table header.
    print(
        f"{'Model':<30} "
        f"{'Params':>10} "
        f"{'Accuracy':>10}"
    )

    print("-" * 50)

    # Teacher performance.
    print(
        f"{'Teacher (large CNN)':<30} "
        f"{teacher_params:>10,} "
        f"{teacher_acc:>9.2f}%"
    )

    # Baseline student performance.
    print(
        f"{'Student (from scratch)':<30} "
        f"{student_params:>10,} "
        f"{baseline_acc:>9.2f}%"
    )

    # Distilled student performance.
    print(
        f"{'Student (distilled)':<30} "
        f"{student_params:>10,} "
        f"{distilled_acc:>9.2f}%"
    )

    # Measure the benefit of knowledge distillation.
    print(
        f"\nDistillation improvement: "
        f"{distilled_acc - baseline_acc:+.2f}% "
        f"over baseline"
    )

    # Measure remaining gap between teacher and student.
    print(
        f"Teacher-student gap: "
        f"{teacher_acc - distilled_acc:.2f}%"
    )


# ============================================================
# Run the Complete Experiment
# ============================================================
#
# Execute the entire teacher-student knowledge
# distillation pipeline.
run_distillation()

Using device: cuda


100%|██████████| 170M/170M [14:50<00:00, 192kB/s]  



Step 1: Training the Teacher
Teacher parameters: 2,474,506
  Epoch 5/20, Loss: 0.5402
  Epoch 10/20, Loss: 0.2766
  Epoch 15/20, Loss: 0.1349
  Epoch 20/20, Loss: 0.0816
Teacher test accuracy: 82.32%

Step 2: Distilling Knowledge into the Student
Student parameters: 152,074
Compression ratio: 16.3x
  Epoch 5/20, Loss: 3.3141
  Epoch 10/20, Loss: 2.6027
  Epoch 15/20, Loss: 2.2221
  Epoch 20/20, Loss: 1.9840
Distilled student test accuracy: 77.37%

Step 3: Training Baseline Student (from scratch)
  Epoch 5/20, Loss: 0.8389
  Epoch 10/20, Loss: 0.6893
  Epoch 15/20, Loss: 0.5885
  Epoch 20/20, Loss: 0.5235
Baseline student test accuracy: 76.98%

RESULTS SUMMARY
Model                              Params   Accuracy
--------------------------------------------------
Teacher (large CNN)             2,474,506     82.32%
Student (from scratch)            152,074     76.98%
Student (distilled)               152,074     77.37%

Distillation improvement: +0.39% over baseline
Teacher-student gap: